In [ ]:
import os
import pandas as pd
import streamlit as st
import os

In [ ]:
base_path = r"C:/Users/efeor/Desktop/PROJECTS/CineSuggest AI/Data/"

movie = pd.read_csv(os.path.join(base_path, "movie.csv"))
tag = pd.read_csv(os.path.join(base_path, "tag.csv"))
genome_tags = pd.read_csv(os.path.join(base_path, "genome_tags.csv"))
link = pd.read_csv(os.path.join(base_path, "link.csv"))

merged_df = pd.merge(movie, link, on="movieId", how="inner")
merged_df = pd.merge(merged_df, tag, on="movieId", how="left")
merged_df = pd.merge(merged_df, genome_tags, on="tag", how="left")

merged_df.drop_duplicates(inplace=True)

merged_df["tag"] = merged_df["tag"].fillna("no tag")
merged_df["tagId"] = merged_df["tagId"].fillna(-1).astype(int)

if "timestamp" in merged_df.columns:
    merged_df["timestamp"] = pd.to_datetime(merged_df["timestamp"])

merged_df["title"] = merged_df["title"].str.strip()

output_file = os.path.join(base_path, "merged_cleaned_movies.csv")
merged_df.to_csv(output_file, index=False)

display(merged_df.head())

,movieId,title,genres,imdbId,tmdbId,userId,tag,timestamp,tagId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1644.0,Watched,2014-12-04 23:44:40,-1
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1741.0,computer animation,2007-07-08 13:59:15,244
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1741.0,Disney animated feature,2007-07-08 22:21:47,-1
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1741.0,Pixar animation,2007-07-08 22:46:10,-1
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862.0,1741.0,TÃ©a Leoni does not star in this movie,2009-06-15 19:19:33,-1


In [ ]:
st.set_page_config(page_title="CineSuggest AI", layout="wide")

@st.cache_data
def get_processed_data():
    file_path = r'C:/Users/efeor/Desktop/PROJECTS/CineSuggest AI/Data/merged_cleaned_movies.csv'
    
    if os.path.exists(file_path):
        return pd.read_csv(file_path)
    else:
        base_path = r'C:/Users/efeor/Desktop/PROJECTS/CineSuggest AI/Data/'
        movie = pd.read_csv(os.path.join(base_path, 'movie.csv'))
        tag = pd.read_csv(os.path.join(base_path, 'tag.csv'))
        link = pd.read_csv(os.path.join(base_path, 'link.csv'))
        genome_tags = pd.read_csv(os.path.join(base_path, 'genome_tags.csv'))
        
        merged_df = pd.merge(movie, link, on='movieId', how='inner')
        merged_df = pd.merge(merged_df, tag, on='movieId', how='left')
        merged_df = pd.merge(merged_df, genome_tags, on='tag', how='left')
        
        merged_df.drop_duplicates(inplace=True)
        merged_df['tag'] = merged_df['tag'].fillna('no tag')
        merged_df['tagId'] = merged_df['tagId'].fillna(-1).astype(int)
        merged_df['title'] = merged_df['title'].str.strip()
        return merged_df

def get_recommendations(df, user_id, n=10):
    user_data = df[df['userId'] == user_id]
    
    if user_data.empty:
        popular = df.groupby('title').size().sort_values(ascending=False).head(n)
        return popular.index.tolist(), "Popüler Filmler (Cold Start Aktivasyonu)"
    
    top_genre = user_data['genres'].str.split('|').explode().mode()[0]
    
    recommendations = df[df['genres'].str.contains(top_genre)]
    recommendations = recommendations[~recommendations['movieId'].isin(user_data['movieId'])]
    
    res = recommendations.groupby('title').size().sort_values(ascending=False).head(n)
    return res.index.tolist(), f"Kişiselleştirilmiş Algoritma ({top_genre} Türü Odaklı)"

def main():
    st.title("🎬 CineSuggest AI — Akıllı Film Öneri Sistemi")
    st.write("---")
    
    try:
        df = get_processed_data()
        
        st.sidebar.header("Müşteri / Kullanıcı Simülasyonu")
        user_id = st.sidebar.number_input("Kullanıcı ID (User ID):", min_value=1, value=1741)
        
        if st.sidebar.button("Yapay Zeka Önerilerini Üret"):
            rec_list, method = get_recommendations(df, user_id)
            
            st.subheader(f"👤 Kullanıcı {user_id} İçin Üretilen Sonuçlar")
            st.info(f"💡 **Sistem Çalışma Stratejisi:** {method}")
            
            cols = st.columns(2)
            for i, movie_title in enumerate(rec_list):
                with cols[i % 2]:
                    st.success(f"**{i+1}.** {movie_title}")
                    
    except Exception as e:
        st.error(f"Sistem yüklenirken bir hata oluştu: {e}")

if __name__ == "__main__":
    main()